# Unit19_PINN_Overview | 物理資訊神經網路(PINN)基礎範例

本 Notebook 以 **TensorFlow/Keras 從零實作一個物理資訊神經網路(PINN)**，求解化工最基本的 **一級反應動力學 ODE**，不需下載任何外部資料集。

## 目標
- 用神經網路表示微分方程的解，並以**自動微分**計算導數組成物理殘差
- **正問題**：在完全沒有資料的情況下，僅憑物理方程與初始條件求解濃度曲線，並與解析解比對
- **反問題**：給定少量帶雜訊的量測，將速率常數 $k$ 設為可訓練變數，同時求解並辨識 $k$

控制方程（批次反應器一級反應）：

$$
\frac{dC}{dt} = -kC, \quad C(0)=C_0
$$

其解析解為 $C(t) = C_0\, e^{-kt}$ ，方便驗證 PINN 的正確性。

---
### 0. 環境設定
- 本範例使用合成資料，無需額外下載資料集。
- 使用 TensorFlow/Keras 從零實作 PINN，資料量小，可在 Colab 或本地 CPU 環境執行。

In [1]:
from pathlib import Path
import tensorflow as tf
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'P4_Unit19_PINN_Overview'
SOURCE_DATA_DIR = 'pinn'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

if IN_COLAB:
    shortcut_path = '/content/ChemE-3590'
    try:
        os.remove(shortcut_path)
    except FileNotFoundError:
        pass
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3590')
    os.symlink(source_path, shortcut_path)
    NOTEBOOK_DIR = Path(shortcut_path) / 'Part_4' / 'Unit19'
else:
    NOTEBOOK_DIR = Path.cwd()

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
MODEL_DIR = OUTPUT_DIR / 'models'
FIG_DIR = OUTPUT_DIR / 'figs'
for d in (OUTPUT_DIR, MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 模型輸出目錄: {MODEL_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

print(f"\nTensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ 偵測到 GPU：{gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("△ 未偵測到 GPU，將使用 CPU（本範例資料量小，CPU 即可）")

✓ 偵測到 Local 環境

✓ Notebook工作目錄: D:\MyGit\ChemE-3590\Part_4\Unit19
✓ 結果輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_PINN_Overview
✓ 模型輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_PINN_Overview\models
✓ 圖檔輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_PINN_Overview\figs

TensorFlow Version: 2.10.1
✓ 偵測到 GPU：/physical_device:GPU:0


---
### 1. 導入必要套件

In [2]:
# 基礎套件
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow版本: {tf.__version__}")
print(f"Keras版本: {keras.__version__}")

# 設定隨機種子以確保結果可重現
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.rcParams['axes.unicode_minus'] = False

TensorFlow版本: 2.10.1
Keras版本: 2.10.0


---
### 2. 問題定義：一級反應動力學

我們求解批次反應器中的一級不可逆反應 $A \rightarrow B$ ，其濃度隨時間的變化遵循：

$$
\frac{dC}{dt} = -kC, \quad C(0)=C_0
$$

設定真實參數 $k_{\mathrm{true}}=1.0$ 、初始濃度 $C_0=1.0$ ，求解區間 $t \in [0, 2]$ 。
其**解析解**為 $C(t)=C_0\,e^{-kt}$ ，稍後用來驗證 PINN 的結果。

In [3]:
# 真實物理參數
K_TRUE = 1.0      # 速率常數 k
C0 = 1.0          # 初始濃度
T_MIN, T_MAX = 0.0, 2.0

# 解析解（用於驗證）
def analytical_C(t, k=K_TRUE, c0=C0):
    return c0 * np.exp(-k * t)

t_plot = np.linspace(T_MIN, T_MAX, 200).reshape(-1, 1)
C_exact = analytical_C(t_plot)

plt.figure(figsize=(7, 4.5))
plt.plot(t_plot, C_exact, 'k-', lw=2, label='Analytical  $C(t)=C_0 e^{-kt}$')
plt.xlabel('Time  t')
plt.ylabel('Concentration  C')
plt.title('First-Order Reaction Kinetics (Ground Truth)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'analytical_solution.png', dpi=120)
plt.show()

print(f"k_true = {K_TRUE}, C0 = {C0}")
print(f"C(0) = {analytical_C(0.0):.4f}, C(1) = {analytical_C(1.0):.4f}, C(2) = {analytical_C(2.0):.4f}")

k_true = 1.0, C0 = 1.0
C(0) = 1.0000, C(1) = 0.3679, C(2) = 0.1353


---
### 3. 建立 PINN 模型

PINN 的「解」就是一個以**時間 $t$ 為輸入、濃度 $C$ 為輸出**的神經網路 $\hat{C}_\theta(t)$ 。
我們使用平滑的 $\tanh$ 激活函數（其高階導數不恆為零，適合計算微分方程殘差）。

In [4]:
def build_pinn(n_hidden=4, n_units=20, seed=SEED):
    """建立一個以 t 為輸入、C 為輸出的 PINN (MLP + tanh)"""
    init = keras.initializers.GlorotNormal(seed=seed)
    inp = keras.Input(shape=(1,), name='t')
    x = inp
    for i in range(n_hidden):
        x = layers.Dense(n_units, activation='tanh',
                         kernel_initializer=init, name=f'hidden_{i+1}')(x)
    out = layers.Dense(1, kernel_initializer=init, name='C')(x)
    return keras.Model(inp, out, name='PINN')

pinn_demo = build_pinn()
pinn_demo.summary()

Model: "PINN"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 t (InputLayer)              [(None, 1)]               0         


 hidden_1 (Dense)            (None, 20)                40        


 hidden_2 (Dense)            (None, 20)                420       


... (3 outputs left out)

Total params: 1,321


Trainable params: 1,321


Non-trainable params: 0


_________________________________________________________________


---
### 4. 正問題：自動微分與物理殘差

**正問題**：已知方程與參數，**不使用任何觀測資料**，僅憑物理求解。

關鍵是用 `tf.GradientTape` 對**輸入 $t$** 自動微分，得到 $\dfrac{d\hat{C}}{dt}$ ，再組成物理殘差：

$$
r(t) = \frac{d\hat{C}_\theta}{dt} + k\,\hat{C}_\theta(t)
$$

若網路滿足方程，則 $r(t)\to 0$ 。

In [5]:
def physics_residual(model, t, k):
    """物理殘差 r(t) = dC/dt + k*C，以自動微分計算 dC/dt"""
    with tf.GradientTape() as tape:
        tape.watch(t)
        C = model(t)
    dC_dt = tape.gradient(C, t)
    return dC_dt + k * C

# 配點(collocation points)：在求解域內均勻採樣，無需網格
N_COLLOC = 100
t_colloc = tf.convert_to_tensor(
    np.linspace(T_MIN, T_MAX, N_COLLOC).reshape(-1, 1), dtype=tf.float32)
t_ic = tf.constant([[0.0]], dtype=tf.float32)   # 初始條件位置

print(f"配點數量: {N_COLLOC}，範圍 t ∈ [{T_MIN}, {T_MAX}]")

配點數量: 100，範圍 t ∈ [0.0, 2.0]


### 4.1 訓練正問題 PINN

總損失 = 物理殘差損失 + 初始條件損失（**無資料項**）：

$$
\mathcal{L} = \underbrace{\frac{1}{N_r}\sum_i |r(t_i)|^2}_{\text{physics}} + \underbrace{|\hat{C}_\theta(0)-C_0|^2}_{\text{IC}}
$$

In [6]:
pinn_fwd = build_pinn()
opt_fwd = keras.optimizers.Adam(learning_rate=1e-3)

@tf.function
def train_step_forward():
    with tf.GradientTape() as tape:
        r = physics_residual(pinn_fwd, t_colloc, K_TRUE)
        loss_phys = tf.reduce_mean(tf.square(r))
        loss_ic = tf.reduce_mean(tf.square(pinn_fwd(t_ic) - C0))
        loss = loss_phys + loss_ic
    grads = tape.gradient(loss, pinn_fwd.trainable_variables)
    opt_fwd.apply_gradients(zip(grads, pinn_fwd.trainable_variables))
    return loss, loss_phys, loss_ic

EPOCHS_FWD = 5000
hist_fwd = []
for ep in range(EPOCHS_FWD + 1):
    loss, lp, lic = train_step_forward()
    if ep % 500 == 0:
        hist_fwd.append(float(loss))
        print(f"Epoch {ep:5d} | total={float(loss):.3e} | physics={float(lp):.3e} | IC={float(lic):.3e}")

Epoch     0 | total=1.586e+00 | physics=5.859e-01 | IC=1.000e+00


Epoch   500 | total=4.841e-04 | physics=4.826e-04 | IC=1.515e-06


Epoch  1000 | total=1.637e-04 | physics=1.632e-04 | IC=4.204e-07


Epoch  1500 | total=3.935e-05 | physics=3.932e-05 | IC=3.438e-08


Epoch  2000 | total=1.235e-05 | physics=1.228e-05 | IC=7.491e-08


Epoch  2500 | total=7.536e-06 | physics=7.536e-06 | IC=9.095e-11


Epoch  3000 | total=5.053e-06 | physics=5.040e-06 | IC=1.356e-08


Epoch  3500 | total=4.533e-06 | physics=4.287e-06 | IC=2.453e-07


Epoch  4000 | total=1.056e-05 | physics=1.006e-05 | IC=4.975e-07


Epoch  4500 | total=1.034e-05 | physics=9.665e-06 | IC=6.795e-07


Epoch  5000 | total=1.528e-06 | physics=1.468e-06 | IC=6.048e-08


### 4.2 正問題結果與解析解比較

PINN 在**完全沒有資料**的情況下，僅憑物理方程就重建了濃度曲線。我們與解析解比對並計算相對 $L_2$ 誤差。

In [7]:
C_pinn = pinn_fwd(tf.convert_to_tensor(t_plot, dtype=tf.float32)).numpy()

rel_l2 = np.linalg.norm(C_pinn - C_exact) / np.linalg.norm(C_exact)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(t_plot, C_exact, 'k-', lw=2, label='Analytical')
axes[0].plot(t_plot, C_pinn, 'r--', lw=2, label='PINN (forward)')
axes[0].set_xlabel('Time  t'); axes[0].set_ylabel('Concentration  C')
axes[0].set_title(f'Forward PINN vs Analytical  (rel. $L_2$ = {rel_l2:.2e})')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(t_plot, np.abs(C_pinn - C_exact), 'b-')
axes[1].set_xlabel('Time  t'); axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Pointwise Absolute Error')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'forward_pinn_result.png', dpi=120)
plt.show()

print(f"相對 L2 誤差: {rel_l2:.3e}")
print(f"最大絕對誤差: {np.max(np.abs(C_pinn - C_exact)):.3e}")

相對 L2 誤差: 3.718e-04
最大絕對誤差: 3.604e-04


---
### 5. 反問題：速率常數辨識

**反問題**：已知方程的**形式**（一級反應），但速率常數 $k$ **未知**。我們只有少量**帶雜訊的濃度量測**。
PINN 的做法是把 $k$ 設為**可訓練變數**，與網路權重一起優化，同時求解並辨識 $k$ 。

總損失 = 資料損失 + 物理殘差損失（含未知 $k$ ）+ 初始條件：

$$
\mathcal{L} = \mathcal{L}_{\mathrm{data}} + \mathcal{L}_{\mathrm{physics}}(k) + \mathcal{L}_{\mathrm{IC}}
$$

In [8]:
# 產生少量帶雜訊的觀測資料
rng = np.random.default_rng(SEED)
N_DATA = 12
t_obs = np.linspace(T_MIN, T_MAX, N_DATA).reshape(-1, 1)
noise_std = 0.03
C_obs = analytical_C(t_obs) + rng.normal(0, noise_std, size=t_obs.shape)

t_obs_tf = tf.convert_to_tensor(t_obs, dtype=tf.float32)
C_obs_tf = tf.convert_to_tensor(C_obs, dtype=tf.float32)

plt.figure(figsize=(7, 4.5))
plt.plot(t_plot, C_exact, 'k-', lw=2, label='True solution')
plt.plot(t_obs, C_obs, 'ro', ms=7, label=f'Noisy data (N={N_DATA})')
plt.xlabel('Time  t'); plt.ylabel('Concentration  C')
plt.title('Sparse Noisy Observations for Inverse Problem')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'inverse_data.png', dpi=120)
plt.show()

print(f"觀測資料點數: {N_DATA}，雜訊標準差: {noise_std}")

觀測資料點數: 12，雜訊標準差: 0.03


### 5.1 訓練反問題 PINN

初始猜測 $k_{\mathrm{init}}=0.3$ （遠離真值 1.0）。訓練過程中追蹤 $k$ 的收斂。

In [9]:
pinn_inv = build_pinn()
k_var = tf.Variable(0.3, dtype=tf.float32, name='k')   # 待辨識的參數
opt_inv = keras.optimizers.Adam(learning_rate=5e-3)

@tf.function
def train_step_inverse():
    with tf.GradientTape() as tape:
        # 資料損失
        C_pred = pinn_inv(t_obs_tf)
        loss_data = tf.reduce_mean(tf.square(C_pred - C_obs_tf))
        # 物理殘差（含未知 k_var）
        r = physics_residual(pinn_inv, t_colloc, k_var)
        loss_phys = tf.reduce_mean(tf.square(r))
        # 初始條件
        loss_ic = tf.reduce_mean(tf.square(pinn_inv(t_ic) - C0))
        loss = loss_data + loss_phys + loss_ic
    var_list = pinn_inv.trainable_variables + [k_var]
    grads = tape.gradient(loss, var_list)
    opt_inv.apply_gradients(zip(grads, var_list))
    return loss, loss_data, loss_phys, k_var

EPOCHS_INV = 5000
k_history = []
for ep in range(EPOCHS_INV + 1):
    loss, ld, lp, kv = train_step_inverse()
    if ep % 250 == 0:
        k_history.append((ep, float(kv)))
    if ep % 500 == 0:
        print(f"Epoch {ep:5d} | total={float(loss):.3e} | data={float(ld):.3e} | "
              f"physics={float(lp):.3e} | k={float(kv):.4f}")

print(f"\n✓ 辨識結果: k_pred = {float(k_var):.4f}  (真值 k_true = {K_TRUE})")
print(f"  相對誤差: {abs(float(k_var) - K_TRUE)/K_TRUE*100:.2f}%")

Epoch     0 | total=1.461e+00 | data=2.356e-01 | physics=2.256e-01 | k=0.2950


Epoch   500 | total=8.611e-04 | data=7.529e-04 | physics=9.982e-05 | k=1.0056


Epoch  1000 | total=8.740e-04 | data=7.912e-04 | physics=7.433e-05 | k=1.0190


Epoch  1500 | total=7.357e-04 | data=7.208e-04 | physics=1.487e-05 | k=1.0195


Epoch  2000 | total=7.297e-04 | data=7.165e-04 | physics=1.306e-05 | k=1.0201


Epoch  2500 | total=7.564e-04 | data=7.332e-04 | physics=1.942e-05 | k=1.0203


Epoch  3000 | total=7.387e-04 | data=7.208e-04 | physics=1.537e-05 | k=1.0201


Epoch  3500 | total=7.275e-04 | data=7.132e-04 | physics=1.426e-05 | k=1.0202


Epoch  4000 | total=1.022e-03 | data=7.900e-04 | physics=1.873e-04 | k=1.0155


Epoch  4500 | total=1.205e-03 | data=8.385e-04 | physics=2.854e-04 | k=1.0198


Epoch  5000 | total=7.274e-04 | data=7.107e-04 | physics=1.669e-05 | k=1.0203

✓ 辨識結果: k_pred = 1.0203  (真值 k_true = 1.0)
  相對誤差: 2.03%


### 5.2 反問題結果： $k$ 收斂與解的擬合

左圖顯示可訓練參數 $k$ 從初始猜測收斂到真值；右圖顯示 PINN 解同時擬合了稀疏資料並符合物理。

In [10]:
k_eps = np.array([h[0] for h in k_history])
k_vals = np.array([h[1] for h in k_history])
C_inv = pinn_inv(tf.convert_to_tensor(t_plot, dtype=tf.float32)).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(k_eps, k_vals, 'b-o', ms=3, label='Identified k')
axes[0].axhline(K_TRUE, color='k', ls='--', label=f'True k = {K_TRUE}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('k')
axes[0].set_title('Convergence of Identified Rate Constant')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(t_plot, C_exact, 'k-', lw=2, label='True solution')
axes[1].plot(t_plot, C_inv, 'g--', lw=2, label='PINN (inverse)')
axes[1].plot(t_obs, C_obs, 'ro', ms=6, label='Noisy data')
axes[1].set_xlabel('Time  t'); axes[1].set_ylabel('Concentration  C')
axes[1].set_title('Inverse PINN Fit')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'inverse_pinn_result.png', dpi=120)
plt.show()

print(f"最終 k_pred = {float(k_var):.4f} | k_true = {K_TRUE}")

最終 k_pred = 1.0203 | k_true = 1.0


---
### 6. 模型權重儲存

In [11]:
# 儲存正問題與反問題 PINN 權重 (TensorFlow checkpoint 格式)
fwd_path = MODEL_DIR / 'pinn_forward'
inv_path = MODEL_DIR / 'pinn_inverse'
pinn_fwd.save_weights(str(fwd_path))
pinn_inv.save_weights(str(inv_path))

# 儲存辨識出的參數
import json
with open(MODEL_DIR / 'identified_params.json', 'w', encoding='utf-8') as f:
    json.dump({'k_true': K_TRUE, 'k_pred': float(k_var), 'C0': C0}, f,
              ensure_ascii=False, indent=2)

print(f"✓ 正問題權重儲存: {fwd_path.name}")
print(f"✓ 反問題權重儲存: {inv_path.name}")
print(f"✓ 辨識參數儲存: identified_params.json")

✓ 正問題權重儲存: pinn_forward
✓ 反問題權重儲存: pinn_inverse
✓ 辨識參數儲存: identified_params.json


---
### 7. 總結

本 Notebook 完成了以下內容：

1. **問題定義**：批次反應器一級反應動力學 ODE，具解析解 $C(t)=C_0 e^{-kt}$
2. **PINN 模型**：以 $\tanh$ MLP 將時間 $t$ 映射到濃度 $C$ ，整個網路對輸入可微
3. **自動微分**：用 `tf.GradientTape` 對輸入 $t$ 求導，組成物理殘差 $r=\dfrac{dC}{dt}+kC$
4. **正問題**：在**無任何資料**下，僅憑物理殘差 + 初始條件訓練，成功逼近解析解
5. **反問題**：把速率常數 $k$ 設為**可訓練變數**，從稀疏帶雜訊資料同時求解並辨識 $k \approx k_{\mathrm{true}}$
6. **複合損失**：體會資料損失、物理殘差、初始條件如何協同約束解

**關鍵啟示**：PINN 把「物理方程」當作一種強力的正則化，使模型在資料稀少時仍遵守物理定律，並以同一框架優雅地處理正問題與反問題（參數辨識）。掌握此從零實作後，再使用 DeepXDE 等高階工具會非常容易。